# VariousMetrics Module - Quick Start Guide

This notebook demonstrates how to use the enhanced VariousMetrics module for microbial community coalescence analysis.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import the enhanced VariousMetrics module
import VariousMetrics as vm

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

print("VariousMetrics module loaded successfully!")
print(f"Available metrics: {len(vm.list_available_metrics()['similarity_metrics'])} similarity, {len(vm.list_available_metrics()['diversity_metrics'])} diversity")

## 1. Basic Diversity and Similarity Analysis

In [ ]:
# Create sample microbial communities
community_A = np.array([20, 15, 10, 8, 5, 3, 2, 1])  # High diversity
community_B = np.array([35, 5, 4, 3, 2, 1, 1, 1])    # Low diversity (dominated)
community_C = np.array([8, 8, 8, 8, 8, 8, 4, 4])     # High evenness

communities = {'High Diversity': community_A, 'Low Diversity': community_B, 'High Evenness': community_C}

# Calculate diversity metrics for each community
diversity_results = {}
for name, community in communities.items():
    diversity_results[name] = vm.calculate_all_ecological_metrics(community)

# Display results
diversity_df = pd.DataFrame(diversity_results).T
print("Diversity Metrics Comparison:")
print(diversity_df[['richness', 'shannon', 'simpson', 'evenness_pielou', 'dominance_berger_parker']].round(3))

In [ ]:
# Visualize diversity metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

metrics_to_plot = ['shannon', 'simpson', 'evenness_pielou', 'dominance_berger_parker']
titles = ['Shannon Diversity', 'Simpson Diversity', 'Pielou Evenness', 'Berger-Parker Dominance']

for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    ax = axes[i//2, i%2]
    values = [diversity_results[name][metric] for name in communities.keys()]
    bars = ax.bar(communities.keys(), values, alpha=0.7)
    ax.set_title(title)
    ax.set_ylabel('Value')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 2. Similarity Analysis Between Communities

In [ ]:
# Calculate all similarity metrics between communities
similarity_matrix = {}
community_names = list(communities.keys())

for metric in ['bray_curtis', 'jensen_shannon', 'jaccard', 'cosine', 'euclidean']:
    matrix = np.zeros((len(communities), len(communities)))
    for i, name1 in enumerate(community_names):
        for j, name2 in enumerate(community_names):
            similarity_func = vm.get_similarity_function(metric)
            matrix[i, j] = similarity_func(communities[name1], communities[name2])
    similarity_matrix[metric] = matrix

# Plot similarity matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, metric in enumerate(['bray_curtis', 'jensen_shannon', 'jaccard', 'cosine', 'euclidean']):
    im = axes[i].imshow(similarity_matrix[metric], cmap='viridis', vmin=0, vmax=1)
    axes[i].set_title(f'{metric.replace("_", " ").title()} Similarity')
    axes[i].set_xticks(range(len(community_names)))
    axes[i].set_yticks(range(len(community_names)))
    axes[i].set_xticklabels(community_names, rotation=45)
    axes[i].set_yticklabels(community_names)
    
    # Add text annotations
    for row in range(len(community_names)):
        for col in range(len(community_names)):
            axes[i].text(col, row, f'{similarity_matrix[metric][row, col]:.3f}',
                        ha='center', va='center', color='white', fontsize=8)
    
    plt.colorbar(im, ax=axes[i])

# Remove empty subplot
axes[5].remove()

plt.tight_layout()
plt.show()

## 3. Coalescence Event Analysis

In [ ]:
# Simulate a coalescence event
np.random.seed(42)

# Parent communities
parent1 = np.array([25, 15, 10, 8, 5, 3, 2, 1])  # Parent 1
parent2 = np.array([5, 8, 12, 15, 10, 6, 3, 2])  # Parent 2

# Offspring (mixture of parents with some stochasticity)
offspring = 0.6 * parent1 + 0.4 * parent2 + np.random.poisson(2, 8)
offspring = offspring.astype(int)

print("Coalescence Event Data:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")
print(f"Offspring: {offspring}")

# Comprehensive coalescence analysis
coalescence_result = vm.analyze_coalescence_event(
    offspring, parent1, parent2,
    metrics=['bray_curtis', 'jensen_shannon', 'jaccard'],
    include_vector_decomp=True,
    include_additivity=True
)

print("\nCoalescence Analysis Results:")
print("-" * 40)

In [ ]:
# Display similarity results
print("Similarity to Parents:")
for metric in ['bray_curtis', 'jensen_shannon', 'jaccard']:
    sim_data = coalescence_result['similarities'][metric]
    print(f"\n{metric.replace('_', ' ').title()}:")
    print(f"  To Parent 1: {sim_data['to_parent1']:.3f}")
    print(f"  To Parent 2: {sim_data['to_parent2']:.3f}")
    print(f"  Parent 1 to Parent 2: {sim_data['parent1_to_parent2']:.3f}")

print("\nRelative Similarity Analysis:")
for metric in ['bray_curtis', 'jensen_shannon', 'jaccard']:
    rel_data = coalescence_result['relative_similarities'][metric]
    print(f"\n{metric.replace('_', ' ').title()}:")
    print(f"  Relative similarity to Parent 1: {rel_data['relative_similarity_parent1']:.3f}")
    print(f"  Relative similarity to Parent 2: {rel_data['relative_similarity_parent2']:.3f}")
    print(f"  Deviation from equal (0.5): {rel_data['deviation_from_equal']:.3f}")

print("\nAdditivity Metrics:")
additivity = coalescence_result['additivity']
for metric, value in additivity.items():
    print(f"  {metric}: {value:.3f}")

In [ ]:
# Visualize coalescence results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Absolute similarities
metrics = ['bray_curtis', 'jensen_shannon', 'jaccard']
parent1_sims = [coalescence_result['similarities'][m]['to_parent1'] for m in metrics]
parent2_sims = [coalescence_result['similarities'][m]['to_parent2'] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

axes[0, 0].bar(x - width/2, parent1_sims, width, label='Parent 1', alpha=0.7)
axes[0, 0].bar(x + width/2, parent2_sims, width, label='Parent 2', alpha=0.7)
axes[0, 0].set_xlabel('Similarity Metric')
axes[0, 0].set_ylabel('Similarity Value')
axes[0, 0].set_title('Absolute Similarities to Parents')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels([m.replace('_', ' ').title() for m in metrics], rotation=45)
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 1)

# 2. Relative similarities
rel_sims_p1 = [coalescence_result['relative_similarities'][m]['relative_similarity_parent1'] for m in metrics]
rel_sims_p2 = [coalescence_result['relative_similarities'][m]['relative_similarity_parent2'] for m in metrics]

axes[0, 1].bar(x - width/2, rel_sims_p1, width, label='Parent 1', alpha=0.7)
axes[0, 1].bar(x + width/2, rel_sims_p2, width, label='Parent 2', alpha=0.7)
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Equal similarity')
axes[0, 1].set_xlabel('Similarity Metric')
axes[0, 1].set_ylabel('Relative Similarity')
axes[0, 1].set_title('Relative Similarities to Parents')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([m.replace('_', ' ').title() for m in metrics], rotation=45)
axes[0, 1].legend()
axes[0, 1].set_ylim(0, 1)

# 3. Community composition
species = np.arange(len(parent1))
axes[1, 0].bar(species - 0.2, parent1, 0.2, label='Parent 1', alpha=0.7)
axes[1, 0].bar(species, parent2, 0.2, label='Parent 2', alpha=0.7)
axes[1, 0].bar(species + 0.2, offspring, 0.2, label='Offspring', alpha=0.7)
axes[1, 0].set_xlabel('Species')
axes[1, 0].set_ylabel('Abundance')
axes[1, 0].set_title('Community Composition')
axes[1, 0].legend()

# 4. Deviations from equal similarity
deviations = [coalescence_result['relative_similarities'][m]['deviation_from_equal'] for m in metrics]
colors = ['green' if d < 0.1 else 'orange' if d < 0.2 else 'red' for d in deviations]

bars = axes[1, 1].bar(range(len(metrics)), deviations, color=colors, alpha=0.7)
axes[1, 1].set_xlabel('Similarity Metric')
axes[1, 1].set_ylabel('Deviation from Equal Similarity')
axes[1, 1].set_title('Bias in Parent Preference')
axes[1, 1].set_xticks(range(len(metrics)))
axes[1, 1].set_xticklabels([m.replace('_', ' ').title() for m in metrics], rotation=45)

# Add value labels
for bar, value in zip(bars, deviations):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                   f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 4. Multiple Events Analysis

In [ ]:
# Simulate multiple coalescence events
np.random.seed(123)
n_events = 20
n_species = 8

offspring_list = []
parent1_list = []
parent2_list = []

for i in range(n_events):
    # Generate diverse parent communities
    p1 = np.random.gamma(2, 5, n_species).astype(int)
    p2 = np.random.gamma(2, 5, n_species).astype(int)
    
    # Offspring as mixture with varying bias
    bias = np.random.uniform(0.3, 0.7)  # Random bias towards parent 1
    offspring = bias * p1 + (1-bias) * p2 + np.random.poisson(1, n_species)
    offspring = offspring.astype(int)
    
    parent1_list.append(p1)
    parent2_list.append(p2)
    offspring_list.append(offspring)

print(f"Generated {n_events} coalescence events")

# Analyze multiple events
population_results = vm.analyze_multiple_coalescence_events(
    offspring_list, parent1_list, parent2_list,
    metrics=['bray_curtis', 'jensen_shannon'],
    include_bias_analysis=True
)

print("\nPopulation-Level Analysis:")
print("-" * 30)

In [ ]:
# Display population statistics
for metric in ['bray_curtis', 'jensen_shannon']:
    stats = population_results['summary_statistics'][metric]
    bias = population_results['bias_analysis'][metric]
    
    print(f"\n{metric.replace('_', ' ').title()} Population Statistics:")
    print(f"  Mean similarity to Parent 1: {stats['similarities_parent1']['mean']:.3f} ± {stats['similarities_parent1']['std']:.3f}")
    print(f"  Mean similarity to Parent 2: {stats['similarities_parent2']['mean']:.3f} ± {stats['similarities_parent2']['std']:.3f}")
    print(f"  Mean relative similarity to Parent 1: {stats['relative_similarities_parent1']['mean']:.3f}")
    print(f"  Mean deviation from equal: {stats['deviations_from_equal']['mean']:.3f}")
    print(f"  Population bias: {bias['bias_direction']} (magnitude: {bias['bias_magnitude']:.3f})")

In [ ]:
# Visualize population results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Prepare data for plotting
for i, metric in enumerate(['bray_curtis', 'jensen_shannon']):
    # Get individual event data
    similarities_p1 = [result['similarities'][metric]['to_parent1'] 
                      for result in population_results['individual_events']]
    similarities_p2 = [result['similarities'][metric]['to_parent2'] 
                      for result in population_results['individual_events']]
    
    # Prepare plotting data
    plot_data = vm.prepare_plotting_data(similarities_p1, similarities_p2, metric)
    
    # Box plot of similarities
    sns.boxplot(data=plot_data, x='Parent', y='Similarity', ax=axes[0, i])
    axes[0, i].set_title(f'{metric.replace("_", " ").title()} Similarities')
    axes[0, i].set_ylim(0, 1)
    
    # Scatter plot showing relationship
    axes[1, i].scatter(similarities_p1, similarities_p2, alpha=0.7, s=50)
    axes[1, i].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Equal similarity line')
    axes[1, i].set_xlabel('Similarity to Parent 1')
    axes[1, i].set_ylabel('Similarity to Parent 2')
    axes[1, i].set_title(f'{metric.replace("_", " ").title()} Parent Comparison')
    axes[1, i].legend()
    axes[1, i].set_xlim(0, 1)
    axes[1, i].set_ylim(0, 1)
    
    # Add correlation info
    corr = np.corrcoef(similarities_p1, similarities_p2)[0, 1]
    axes[1, i].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[1, i].transAxes,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

## 5. Advanced Analysis: Deviation from Neutral

In [ ]:
# Analyze deviations from neutral coalescence (equal similarity to both parents)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, metric in enumerate(['bray_curtis', 'jensen_shannon']):
    # Get relative similarities for this metric
    rel_sims_p1 = [result['relative_similarities'][metric]['relative_similarity_parent1'] 
                   for result in population_results['individual_events']]
    
    # Calculate deviations from 0.5 (neutral)
    deviations = [abs(sim - 0.5) for sim in rel_sims_p1]
    
    # Histogram of relative similarities
    axes[i].hist(rel_sims_p1, bins=15, alpha=0.7, edgecolor='black')
    axes[i].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Neutral (0.5)')
    axes[i].axvline(x=np.mean(rel_sims_p1), color='blue', linestyle='-', linewidth=2, 
                   label=f'Mean ({np.mean(rel_sims_p1):.3f})')
    axes[i].set_xlabel('Relative Similarity to Parent 1')
    axes[i].set_ylabel('Frequency')
    axes[i].set_title(f'{metric.replace("_", " ").title()} Distribution')
    axes[i].legend()
    axes[i].set_xlim(0, 1)
    
    # Add statistics text
    mean_dev = np.mean(deviations)
    frac_above = np.mean(np.array(rel_sims_p1) > 0.5)
    axes[i].text(0.05, 0.95, f'Mean deviation: {mean_dev:.3f}\nFraction > 0.5: {frac_above:.3f}',
                transform=axes[i].transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nNeutrality Analysis:")
print("A perfectly neutral coalescence would show relative similarity = 0.5")
print("Values > 0.5 indicate bias towards Parent 1, < 0.5 indicate bias towards Parent 2")
print("Large deviations from 0.5 suggest non-neutral coalescence processes")

## Summary

This notebook demonstrates the key features of the enhanced VariousMetrics module:

1. **Comprehensive Diversity Metrics**: Shannon, Simpson, Pielou evenness, Berger-Parker dominance, Hill numbers
2. **Multiple Similarity Measures**: Bray-Curtis, Jensen-Shannon, Jaccard, Cosine, Euclidean, Morisita-Horn
3. **Single Event Analysis**: Complete coalescence event characterization with relative similarities and additivity
4. **Population-Level Analysis**: Statistical analysis across multiple events with bias detection
5. **Advanced Visualizations**: Publication-ready plots for all major analysis types

The module provides a unified interface for all coalescence analysis needs while maintaining backward compatibility with existing code.